In [1]:
"""
01_geo_dragnet.ipynb

Queries the NCBI GEO (Gene Expression Omnibus) API to identify and fetch metadata 
for human whole-blood/PBMC transcriptomic datasets related to sepsis. 
Outputs the master menu CSV for downstream harvesting.
"""

import time
import requests
import pandas as pd
from pathlib import Path

# ==========================================
# CONFIGURATION & PATHS
# ==========================================
# NCBI restricts anonymous traffic. Add your email so they don't block the request.
EMAIL = "rkp6055@gmail.com" 
SEARCH_QUERY = 'sepsis[Title] AND "Homo sapiens"[Organism] AND (blood[Description] OR PBMC[Description]) AND "gse"[Filter]'

# Dynamically resolve paths (works regardless of where the repo is cloned)
# Assuming this notebook is running inside the /notebooks/ directory
BASE_DIR = Path.cwd().parent
OUTPUT_DIR = BASE_DIR / "data" / "raw"
OUTPUT_PATH = OUTPUT_DIR / "geo_sepsis_master_menu.csv"

# ==========================================
# PHASE 1: IDENTIFY DATASETS
# ==========================================
print(f"[*] Querying NCBI API for: {SEARCH_QUERY}")

search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "gds",
    "term": SEARCH_QUERY,
    "retmax": 500,
    "retmode": "json",
    "email": EMAIL
}

response = requests.get(search_url, params=params).json()
id_list = response['esearchresult']['idlist']

print(f"[+] Found {len(id_list)} GEO datasets. Ready to fetch metadata...")

[*] Querying NCBI API for: sepsis[Title] AND "Homo sapiens"[Organism] AND (blood[Description] OR PBMC[Description]) AND "gse"[Filter]
[+] Found 121 GEO datasets. Ready to fetch metadata...


In [2]:
# ==========================================
# PHASE 2: METADATA EXTRACTION
# ==========================================
cohort_data = []
summary_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"

print("[*] Fetching metadata in batches (respecting NCBI API rate limits)...")

for i in range(0, len(id_list), 50):
    batch_ids = id_list[i:i+50]
    id_string = ",".join(batch_ids)
    
    sum_params = {
        "db": "gds",
        "id": id_string,
        "retmode": "json",
        "email": EMAIL
    }
    
    sum_resp = requests.get(summary_url, params=sum_params).json()
    
    # Parse the JSON response
    if 'result' in sum_resp:
        for uid in batch_ids:
            if uid in sum_resp['result']:
                doc = sum_resp['result'][uid]
                # We only want the whole Series (GSE), not individual patient samples
                if doc.get('entrytype') == 'GSE':
                    cohort_data.append({
                        'Accession': doc.get('accession', ''),
                        'Title': doc.get('title', ''),
                        'Samples': doc.get('n_samples', 0),
                        'Summary': doc.get('summary', ''),
                        'Type': doc.get('gdstype', '')
                    })
                    
    time.sleep(1) # Be polite to NCBI servers to avoid IP bans
    print(f"    -> Processed {min(i+50, len(id_list))} / {len(id_list)} datasets")

print("\n[+] Metadata harvest complete.")

[*] Fetching metadata in batches (this takes a minute to respect API limits)...
    -> Processed 50 / 121 datasets
    -> Processed 100 / 121 datasets
    -> Processed 121 / 121 datasets

[+] Metadata harvest complete.


In [3]:
# ==========================================
# PHASE 3: FORMAT & EXPORT
# ==========================================
# Create DataFrame
df = pd.DataFrame(cohort_data)

# Convert 'Samples' to numbers so it sorts properly, then sort descending
df['Samples'] = pd.to_numeric(df['Samples'], errors='coerce')
df = df.sort_values(by='Samples', ascending=False)

# Ensure output directory exists and save
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"[*] Exported {len(df)} datasets to: {OUTPUT_PATH}")

# Display the top 10 largest datasets in the notebook for visual verification
df.head(10)

[*] Exported 121 datasets to /workspace/data/raw/geo_sepsis_master_menu.csv


,Accession,Title,Samples,Summary,Type
87,GSE100150,A Transcriptome Fingerprinting Assay for Clini...,985,Development of a cost-effective and practical ...,Expression profiling by array
77,GSE134364,Protein-coding and non-coding RNA landscape in...,901,This SuperSeries is composed of the SubSeries ...,Expression profiling by array
78,GSE134358,Small non-coding RNA expression in whole blood...,567,Genome-wide gene expression profiling of whole...,Expression profiling by array
96,GSE72829,Diagnosis of childhood bacterial and viral inf...,531,This SuperSeries is composed of the SubSeries ...,Expression profiling by array
23,GSE205672,Transcriptomes of human PBMCs and monocytes in...,460,FUSE cohort,Expression profiling by high throughput sequen...
44,GSE236713,Biomarker Validation on the ‘Analysis of geNe ...,447,Early diagnosis of sepsis and discrimination f...,Expression profiling by array
64,GSE185263,Predicting sepsis severity at first clinical p...,392,Identifying at first clinical presentation gen...,Expression profiling by high throughput sequen...
79,GSE134347,Protein-coding and non-coding RNA expression i...,298,Genome-wide gene expression profiling of whole...,Expression profiling by array
34,GSE208581,Risk assessment with gene expression markers i...,267,We investigated the individual phenotypic pred...,Expression profiling by high throughput sequen...
10,GSE279452,Multiomics Analyses Unveil Immune Landscape in...,266,"Sepsis is a leading cause of global mortality,...",Expression profiling by high throughput sequen...
